In [7]:
# Day 3 - Fundamental Scoring System
import pandas as pd
import numpy as np
import requests
import time
import warnings
from bs4 import BeautifulSoup
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [8]:
# ============================================
# SCREENER.IN SCRAPER (from Day 2)
# ============================================

def extract_table(table):
    try:
        thead   = table.find('thead')
        columns = []
        if thead:
            for th in thead.find_all('th'):
                text = th.get_text(strip=True)
                columns.append(text if text else 'Metric')

        tbody = table.find('tbody')
        if not tbody:
            return None

        rows = tbody.find_all('tr')
        data = {}

        for row in rows:
            cells       = row.find_all('td')
            if not cells:
                continue
            metric_name = cells[0].get_text(strip=True)
            metric_name = metric_name.replace('+', '').strip()

            skip_keywords = ['Raw PDF', 'PDF', 'Source']
            if any(kw in metric_name for kw in skip_keywords):
                continue

            values = []
            for cell in cells[1:]:
                val = cell.get_text(strip=True)
                val = val.replace(',', '').replace('%', '').strip()
                try:
                    values.append(float(val))
                except:
                    values.append(val if val else None)

            data[metric_name] = values

        if not data:
            return None

        col_names = columns[1:] if len(columns) > 1 else list(range(len(values)))
        df        = pd.DataFrame(data, index=col_names).T
        return df

    except Exception as e:
        print(f"Table extraction error: {e}")
        return None


def scrape_screener_complete(symbol):
    try:
        url     = f"https://www.screener.in/company/{symbol}/consolidated/"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

        response = requests.get(url, headers=headers, timeout=15)

        # Try standalone if consolidated not available
        if response.status_code != 200:
            url      = f"https://www.screener.in/company/{symbol}/"
            response = requests.get(url, headers=headers, timeout=15)

        if response.status_code != 200:
            return None

        soup     = BeautifulSoup(response.text, 'html.parser')
        sections = soup.find_all('section')

        target_sections = [
            'Quarterly Results',
            'Profit & Loss',
            'Balance Sheet',
            'Cash Flows',
            'Ratios',
            'Shareholding Pattern',
        ]

        result = {}

        for section in sections:
            heading = section.find('h2')
            if not heading:
                continue
            section_name = heading.text.strip()
            if section_name not in target_sections:
                continue
            table = section.find('table')
            if not table:
                continue
            df = extract_table(table)
            if df is not None:
                result[section_name] = df

        # Extract sector info
        for section in sections:
            heading = section.find('h2')
            if heading and 'Peer' in heading.text:
                links       = section.find_all('a')
                sector_info = {}
                titles      = ['Broad Sector', 'Sector', 'Broad Industry', 'Industry']
                for link in links:
                    title = link.get('title', '')
                    if title in titles:
                        sector_info[title] = link.text.strip()
                result['Sector Info'] = sector_info
                break

        return result if result else None

    except Exception as e:
        print(f"Error scraping {symbol}: {e}")
        return None

print("Scraper functions loaded!")

Scraper functions loaded!


In [9]:
# 50 stock sample — diversified across sectors and market caps
stocks_to_fetch = [
    # Large Cap
    'TCS', 'INFY', 'HDFCBANK', 'RELIANCE', 'WIPRO',

    # Mid Cap IT
    'PERSISTENT', 'MPHASIS', 'LTTS', 'COFORGE', 'HAPPSTMNDS',

    # Mid Cap Finance
    'CHOLAFIN', 'MUTHOOTFIN', 'MANAPPURAM', 'IIFL', 'CREDITACC',

    # Small Cap Manufacturing
    'GARFIBRES', 'SUPRAJIT', 'HIMATSEIDE', 'KILPEST', 'IGPL',

    # Defence
    'HAL', 'BEL', 'BHEL', 'MIDHANI', 'PARAS',

    # Specialty Chemicals
    'VINATIORGA', 'CLEAN', 'FINEORG', 'GALAXYSURF', 'AAVAS',

    # Pharma
    'SUNPHARMA', 'DIVISLAB', 'APLLTD', 'GRANULES', 'IPCALAB',

    # Consumer
    'TITAN', 'HAVELLS', 'VGUARD', 'SYMPHONY', 'WONDERLA',

    # Agrochem
    'PIIND', 'RALLIS', 'DHANUKA', 'BAYER', 'INSECTICIDE',

    # Micro Cap
    'DYNPRO', 'HEGDE', 'GULFPETRO', 'SUVIDHAA', 'ABSLAMC',
]

print(f"Total stocks to fetch: {len(stocks_to_fetch)}")

Total stocks to fetch: 50


In [10]:
# Fetch data for all 50 stocks
all_stock_data = {}
failed_stocks  = []

print(f"Starting fetch for {len(stocks_to_fetch)} stocks...")
print("This will take 10-15 minutes. Please wait...\n")

for i, symbol in enumerate(stocks_to_fetch):
    try:
        print(f"[{i+1}/50] Fetching {symbol}...", end=" ")
        
        data = scrape_screener_complete(symbol)
        
        if data:
            all_stock_data[symbol] = data
            sections = [k for k in data.keys() if k != 'Sector Info']
            print(f"✓ ({len(sections)} sections)")
        else:
            failed_stocks.append(symbol)
            print(f"✗ Failed")
        
        # Wait 2 seconds between requests
        # Avoids getting blocked by Screener.in
        time.sleep(2)
        
    except Exception as e:
        failed_stocks.append(symbol)
        print(f"✗ Error: {e}")
        time.sleep(2)

print(f"\n{'='*50}")
print(f"Successfully fetched : {len(all_stock_data)} stocks")
print(f"Failed               : {len(failed_stocks)} stocks")
if failed_stocks:
    print(f"Failed stocks        : {failed_stocks}")

Starting fetch for 50 stocks...
This will take 10-15 minutes. Please wait...

[1/50] Fetching TCS... ✓ (6 sections)
[2/50] Fetching INFY... ✓ (6 sections)
[3/50] Fetching HDFCBANK... ✓ (6 sections)
[4/50] Fetching RELIANCE... ✓ (6 sections)
[5/50] Fetching WIPRO... ✓ (6 sections)
[6/50] Fetching PERSISTENT... ✓ (6 sections)
[7/50] Fetching MPHASIS... ✓ (6 sections)
[8/50] Fetching LTTS... ✓ (6 sections)
[9/50] Fetching COFORGE... ✓ (6 sections)
[10/50] Fetching HAPPSTMNDS... ✓ (6 sections)
[11/50] Fetching CHOLAFIN... ✓ (6 sections)
[12/50] Fetching MUTHOOTFIN... ✓ (6 sections)
[13/50] Fetching MANAPPURAM... ✓ (6 sections)
[14/50] Fetching IIFL... ✓ (6 sections)
[15/50] Fetching CREDITACC... ✓ (6 sections)
[16/50] Fetching GARFIBRES... ✓ (6 sections)
[17/50] Fetching SUPRAJIT... ✓ (6 sections)
[18/50] Fetching HIMATSEIDE... ✓ (6 sections)
[19/50] Fetching KILPEST... ✗ Failed
[20/50] Fetching IGPL... ✓ (6 sections)
[21/50] Fetching HAL... ✓ (6 sections)
[22/50] Fetching BEL... ✓ (6 sect

In [11]:
import pickle

# Save to data folder
with open('../data/raw_stock_data.pkl', 'wb') as f:
    pickle.dump(all_stock_data, f)

print(f"Saved {len(all_stock_data)} stocks to raw_stock_data.pkl")
print("You can reload this anytime without re-scraping")

Saved 46 stocks to raw_stock_data.pkl
You can reload this anytime without re-scraping


In [12]:
# ============================================
# EXTRACT KEY METRICS FROM RAW DATA
# ============================================

def extract_metrics(symbol, data):
    """
    Extract all key fundamental metrics 
    from raw Screener.in data for one stock
    """
    metrics = {'Symbol': symbol}
    
    try:
        # ── SECTOR INFO ──────────────────────────
        sector_info = data.get('Sector Info', {})
        metrics['Sector']       = sector_info.get('Sector', None)
        metrics['Industry']     = sector_info.get('Industry', None)
        metrics['Broad Sector'] = sector_info.get('Broad Sector', None)

        # ── PROFIT & LOSS (Annual) ───────────────
        pl = data.get('Profit & Loss')
        if pl is not None:
            # Remove TTM column for calculations
            pl_clean = pl.drop(columns=['TTM'], errors='ignore')
            
            # Sales (Revenue)
            if 'Sales' in pl_clean.index:
                sales          = pl_clean.loc['Sales'].dropna()
                metrics['Latest Revenue']     = sales.iloc[-1] if len(sales) > 0 else None
                metrics['Revenue 3Y ago']     = sales.iloc[-4] if len(sales) >= 4 else None
                metrics['Revenue 5Y ago']     = sales.iloc[-6] if len(sales) >= 6 else None

                # 3 year revenue CAGR
                if metrics['Latest Revenue'] and metrics['Revenue 3Y ago'] and metrics['Revenue 3Y ago'] > 0:
                    metrics['Revenue CAGR 3Y'] = (
                        (metrics['Latest Revenue'] / metrics['Revenue 3Y ago']) ** (1/3) - 1
                    ) * 100
                else:
                    metrics['Revenue CAGR 3Y'] = None

            # Net Profit
            if 'Net Profit' in pl_clean.index:
                profit         = pl_clean.loc['Net Profit'].dropna()
                metrics['Latest Profit']  = profit.iloc[-1] if len(profit) > 0 else None
                metrics['Profit 3Y ago']  = profit.iloc[-4] if len(profit) >= 4 else None

                # 3 year profit CAGR
                if (metrics['Latest Profit'] and metrics['Profit 3Y ago'] 
                    and metrics['Profit 3Y ago'] > 0):
                    metrics['Profit CAGR 3Y'] = (
                        (metrics['Latest Profit'] / metrics['Profit 3Y ago']) ** (1/3) - 1
                    ) * 100
                else:
                    metrics['Profit CAGR 3Y'] = None

            # Operating Margin trend
            if 'OPM %' in pl_clean.index:
                opm            = pl_clean.loc['OPM %'].dropna()
                metrics['Latest OPM']     = opm.iloc[-1] if len(opm) > 0 else None
                metrics['OPM 3Y ago']     = opm.iloc[-4] if len(opm) >= 4 else None

        # ── BALANCE SHEET ────────────────────────
        bs = data.get('Balance Sheet')
        if bs is not None:
            bs_clean = bs.copy()

            # Borrowings (Debt)
            if 'Borrowings' in bs_clean.index:
                debt           = bs_clean.loc['Borrowings'].dropna()
                metrics['Latest Debt']    = debt.iloc[-1] if len(debt) > 0 else None
                metrics['Debt 3Y ago']    = debt.iloc[-4] if len(debt) >= 4 else None

            # Equity
            if 'Reserves' in bs_clean.index and 'Equity Capital' in bs_clean.index:
                reserves       = bs_clean.loc['Reserves'].dropna()
                equity_cap     = bs_clean.loc['Equity Capital'].dropna()
                metrics['Latest Equity']  = (
                    reserves.iloc[-1] + equity_cap.iloc[-1]
                    if len(reserves) > 0 and len(equity_cap) > 0 else None
                )

            # Debt to Equity
            if metrics.get('Latest Debt') and metrics.get('Latest Equity'):
                if metrics['Latest Equity'] > 0:
                    metrics['Debt to Equity'] = (
                        metrics['Latest Debt'] / metrics['Latest Equity']
                    )
                else:
                    metrics['Debt to Equity'] = None

            # ROE = Net Profit / Equity
            if metrics.get('Latest Profit') and metrics.get('Latest Equity'):
                if metrics['Latest Equity'] > 0:
                    metrics['ROE'] = (
                        metrics['Latest Profit'] / metrics['Latest Equity'] * 100
                    )
                else:
                    metrics['ROE'] = None

        # ── CASH FLOWS ───────────────────────────
        cf = data.get('Cash Flows')
        if cf is not None:
            if 'Cash from Operating Activity' in cf.index:
                op_cf          = cf.loc['Cash from Operating Activity'].dropna()
                metrics['Latest Operating CF'] = op_cf.iloc[-1] if len(op_cf) > 0 else None
                metrics['Operating CF 3Y ago'] = op_cf.iloc[-4] if len(op_cf) >= 4 else None

                # Count positive CF years
                metrics['CF Positive Years'] = int((op_cf > 0).sum())
                metrics['CF Total Years']     = len(op_cf)

        # ── RATIOS ───────────────────────────────
        ratios = data.get('Ratios')
        if ratios is not None:
            if 'ROCE %' in ratios.index:
                roce           = ratios.loc['ROCE %'].dropna()
                metrics['Latest ROCE']    = roce.iloc[-1] if len(roce) > 0 else None
                metrics['ROCE 3Y ago']    = roce.iloc[-4] if len(roce) >= 4 else None

        # ── SHAREHOLDING ─────────────────────────
        sh = data.get('Shareholding Pattern')
        if sh is not None:
            if 'Promoters' in sh.index:
                promoter       = sh.loc['Promoters'].dropna()
                metrics['Latest Promoter Holding'] = (
                    promoter.iloc[-1] if len(promoter) > 0 else None
                )
                metrics['Promoter Holding 1Y ago'] = (
                    promoter.iloc[-5] if len(promoter) >= 5 else None
                )
                # Promoter trend
                if (metrics.get('Latest Promoter Holding') and 
                    metrics.get('Promoter Holding 1Y ago')):
                    metrics['Promoter Trend'] = (
                        metrics['Latest Promoter Holding'] - 
                        metrics['Promoter Holding 1Y ago']
                    )

            if 'FIIs' in sh.index:
                fii            = sh.loc['FIIs'].dropna()
                metrics['FII Holding'] = fii.iloc[-1] if len(fii) > 0 else None

    except Exception as e:
        print(f"Error extracting metrics for {symbol}: {e}")

    return metrics

# Test on one stock first
test = extract_metrics('TCS', all_stock_data['TCS'])
for key, value in test.items():
    print(f"{key:30} : {value}")

Symbol                         : TCS
Sector                         : Information Technology
Industry                       : Computers - Software & Consulting
Broad Sector                   : Information Technology
Latest Revenue                 : 255324.0
Revenue 3Y ago                 : 191754.0
Revenue 5Y ago                 : 156949.0
Revenue CAGR 3Y                : 10.014282114946793
Latest Profit                  : 48797.0
Profit 3Y ago                  : 38449.0
Profit CAGR 3Y                 : 8.268642437843777
Latest OPM                     : 26.0
OPM 3Y ago                     : 28.0
Latest Debt                    : 10932.0
Debt 3Y ago                    : 7688.0
Latest Equity                  : 106415.0
Debt to Equity                 : 0.1027298783066297
ROE                            : 45.85537753136306
Latest Operating CF            : 48908.0
Operating CF 3Y ago            : 39949.0
CF Positive Years              : 12
CF Total Years                 : 12
Latest ROCE      

In [13]:
def extract_all_metrics(symbol, data):
    """
    Extract all fundamental metrics with zero redundancy
    Annual data for long term trends
    Quarterly data for recent momentum
    """
    m = {'Symbol': symbol}

    try:
        # ════════════════════════════════════════
        # SECTOR INFO
        # ════════════════════════════════════════
        sector_info         = data.get('Sector Info', {})
        m['Sector']         = sector_info.get('Sector',       None)
        m['Industry']       = sector_info.get('Industry',     None)
        m['Broad Sector']   = sector_info.get('Broad Sector', None)

        # ════════════════════════════════════════
        # ANNUAL P&L → Long term CAGR only
        # ════════════════════════════════════════
        pl = data.get('Profit & Loss')
        if pl is not None:
            pl_clean = pl.drop(columns=['TTM'], errors='ignore')

            # Revenue CAGR
            if 'Sales' in pl_clean.index:
                sales = pl_clean.loc['Sales'].dropna()

                # 5 year CAGR
                if len(sales) >= 6:
                    m['Revenue CAGR 5Y'] = round(
                        ((sales.iloc[-1] / sales.iloc[-6]) ** (1/5) - 1) * 100, 2
                    )

                # 10 year CAGR
                if len(sales) >= 11:
                    m['Revenue CAGR 10Y'] = round(
                        ((sales.iloc[-1] / sales.iloc[-11]) ** (1/10) - 1) * 100, 2
                    )

                # Long term average OPM
                if 'OPM %' in pl_clean.index:
                    opm                  = pl_clean.loc['OPM %'].dropna()
                    m['Avg OPM 5Y']      = round(opm.iloc[-5:].mean(), 2) if len(opm) >= 5 else None
                    m['Avg OPM 10Y']     = round(opm.iloc[-10:].mean(), 2) if len(opm) >= 10 else None

            # Profit CAGR
            if 'Net Profit' in pl_clean.index:
                profit = pl_clean.loc['Net Profit'].dropna()
                # Only positive profit companies
                if len(profit) >= 6 and profit.iloc[-6] > 0:
                    m['Profit CAGR 5Y']  = round(
                        ((profit.iloc[-1] / profit.iloc[-6]) ** (1/5) - 1) * 100, 2
                    )
                if len(profit) >= 11 and profit.iloc[-11] > 0:
                    m['Profit CAGR 10Y'] = round(
                        ((profit.iloc[-1] / profit.iloc[-11]) ** (1/10) - 1) * 100, 2
                    )

        # ════════════════════════════════════════
        # QUARTERLY DATA → Recent momentum
        # ════════════════════════════════════════
        qr = data.get('Quarterly Results')
        if qr is not None:
            qr_clean = qr.drop(columns=['TTM'], errors='ignore')

            # Revenue recent momentum
            if 'Sales' in qr_clean.index:
                sales_q = qr_clean.loc['Sales'].dropna()

                # TTM Revenue (last 4 quarters)
                if len(sales_q) >= 4:
                    m['TTM Revenue']         = round(sales_q.iloc[-4:].sum(), 2)

                # YoY growth latest quarter
                if len(sales_q) >= 5:
                    latest                   = sales_q.iloc[-1]
                    year_ago                 = sales_q.iloc[-5]
                    if year_ago > 0:
                        m['Revenue YoY Q']   = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )

                # Count consecutive YoY improvements
                yoy_improvements = 0
                for i in range(1, min(5, len(sales_q) - 4)):
                    curr = sales_q.iloc[-i]
                    prev = sales_q.iloc[-i-4]
                    if prev > 0 and curr > prev:
                        yoy_improvements += 1
                    else:
                        break
                m['Revenue Consecutive YoY'] = yoy_improvements

                # Revenue acceleration
                if len(sales_q) >= 9:
                    recent_yoy   = (sales_q.iloc[-1] - sales_q.iloc[-5]) / sales_q.iloc[-5] * 100
                    previous_yoy = (sales_q.iloc[-5] - sales_q.iloc[-9]) / sales_q.iloc[-9] * 100
                    m['Revenue Accelerating'] = recent_yoy > previous_yoy

            # Profit recent momentum
            if 'Net Profit' in qr_clean.index:
                profit_q = qr_clean.loc['Net Profit'].dropna()

                # TTM Profit
                if len(profit_q) >= 4:
                    m['TTM Profit']          = round(profit_q.iloc[-4:].sum(), 2)

                # YoY profit growth
                if len(profit_q) >= 5:
                    latest                   = profit_q.iloc[-1]
                    year_ago                 = profit_q.iloc[-5]
                    if year_ago > 0:
                        m['Profit YoY Q']    = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )

                # Profit positive quarters
                m['Profit Positive Q']       = int((profit_q > 0).sum())
                m['Total Quarters']          = len(profit_q)

            # Margin recent trend
            if 'OPM %' in qr_clean.index:
                opm_q = qr_clean.loc['OPM %'].dropna()

                m['Latest OPM Q']            = opm_q.iloc[-1]  if len(opm_q) > 0 else None
                m['Avg OPM 4Q']              = round(opm_q.iloc[-4:].mean(), 2) if len(opm_q) >= 4 else None

                # Margin improving vs year ago
                if len(opm_q) >= 5:
                    m['Margin Improving']    = bool(opm_q.iloc[-1] >= opm_q.iloc[-5])

            # EPS recent trend
            if 'EPS in Rs' in qr_clean.index:
                eps_q = qr_clean.loc['EPS in Rs'].dropna()

                m['Latest EPS Q']            = eps_q.iloc[-1] if len(eps_q) > 0 else None

                if len(eps_q) >= 5:
                    latest                   = eps_q.iloc[-1]
                    year_ago                 = eps_q.iloc[-5]
                    if year_ago > 0:
                        m['EPS YoY Growth']  = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )

        # ════════════════════════════════════════
        # BALANCE SHEET → Financial structure
        # ════════════════════════════════════════
        bs = data.get('Balance Sheet')
        if bs is not None:
            if 'Borrowings' in bs.index:
                debt                         = bs.loc['Borrowings'].dropna()
                m['Latest Debt']             = debt.iloc[-1]  if len(debt) > 0 else None

                # Debt reducing over 3 years?
                if len(debt) >= 4:
                    m['Debt Reducing']       = bool(debt.iloc[-1] < debt.iloc[-4])

            if 'Reserves' in bs.index and 'Equity Capital' in bs.index:
                reserves                     = bs.loc['Reserves'].dropna()
                eq_cap                       = bs.loc['Equity Capital'].dropna()

                if len(reserves) > 0 and len(eq_cap) > 0:
                    equity                   = reserves.iloc[-1] + eq_cap.iloc[-1]
                    m['Latest Equity']       = round(equity, 2)

                    # ROE
                    if m.get('TTM Profit') and equity > 0:
                        m['ROE']             = round(m['TTM Profit'] / equity * 100, 2)

                    # Debt to Equity
                    if m.get('Latest Debt') is not None and equity > 0:
                        m['Debt to Equity']  = round(m['Latest Debt'] / equity, 2)

        # ════════════════════════════════════════
        # CASH FLOWS → Cash quality
        # ════════════════════════════════════════
        cf = data.get('Cash Flows')
        if cf is not None:
            if 'Cash from Operating Activity' in cf.index:
                op_cf                        = cf.loc['Cash from Operating Activity'].dropna()
                m['Latest Operating CF']     = op_cf.iloc[-1] if len(op_cf) > 0 else None
                m['CF Positive Years']       = int((op_cf > 0).sum())
                m['CF Total Years']          = len(op_cf)

                # FCF growing?
                if len(op_cf) >= 4:
                    m['CF Growing']          = bool(op_cf.iloc[-1] > op_cf.iloc[-4])

        # ════════════════════════════════════════
        # RATIOS → Efficiency
        # ════════════════════════════════════════
        ratios = data.get('Ratios')
        if ratios is not None:
            if 'ROCE %' in ratios.index:
                roce                         = ratios.loc['ROCE %'].dropna()
                m['Latest ROCE']             = roce.iloc[-1] if len(roce) > 0 else None

                # ROCE improving?
                if len(roce) >= 4:
                    m['ROCE Improving']      = bool(roce.iloc[-1] > roce.iloc[-4])

                # Average ROCE 5Y
                if len(roce) >= 5:
                    m['Avg ROCE 5Y']         = round(roce.iloc[-5:].mean(), 2)

        # ════════════════════════════════════════
        # SHAREHOLDING → Ownership signals
        # ════════════════════════════════════════
        sh = data.get('Shareholding Pattern')
        if sh is not None:
            if 'Promoters' in sh.index:
                promoter                     = sh.loc['Promoters'].dropna()
                m['Promoter Holding']        = promoter.iloc[-1] if len(promoter) > 0 else None

                # Change over last 4 quarters
                if len(promoter) >= 5:
                    m['Promoter Change 4Q']  = round(
                        promoter.iloc[-1] - promoter.iloc[-5], 2
                    )

            if 'FIIs' in sh.index:
                fii                          = sh.loc['FIIs'].dropna()
                m['FII Holding']             = fii.iloc[-1] if len(fii) > 0 else None
                if len(fii) >= 5:
                    m['FII Change 4Q']       = round(fii.iloc[-1] - fii.iloc[-5], 2)

            if 'DIIs' in sh.index:
                dii                          = sh.loc['DIIs'].dropna()
                m['DII Holding']             = dii.iloc[-1] if len(dii) > 0 else None
                if len(dii) >= 5:
                    m['DII Change 4Q']       = round(dii.iloc[-1] - dii.iloc[-5], 2)

    except Exception as e:
        print(f"Error extracting {symbol}: {e}")

    return m

# Test on TCS
print("Testing on TCS:")
tcs_metrics = extract_all_metrics('TCS', all_stock_data['TCS'])
for key, value in tcs_metrics.items():
    print(f"  {key:30} : {value}")

Testing on TCS:
  Symbol                         : TCS
  Sector                         : Information Technology
  Industry                       : Computers - Software & Consulting
  Broad Sector                   : Information Technology
  Revenue CAGR 5Y                : 10.22
  Revenue CAGR 10Y               : 10.43
  Avg OPM 5Y                     : 27.0
  Avg OPM 10Y                    : 27.0
  Profit CAGR 5Y                 : 8.5
  Profit CAGR 10Y                : 9.3
  TTM Revenue                    : 260802.0
  Revenue YoY Q                  : 4.87
  Revenue Consecutive YoY        : 4
  Revenue Accelerating           : False
  TTM Profit                     : 47963.0
  Profit YoY Q                   : -13.85
  Profit Positive Q              : 13
  Total Quarters                 : 13
  Latest OPM Q                   : 27.0
  Avg OPM 4Q                     : 26.75
  Margin Improving               : True
  Latest EPS Q                   : 29.45
  EPS YoY Growth                 : 

In [14]:
# Extract metrics for all 46 stocks
all_metrics = []

for symbol, data in all_stock_data.items():
    metrics = extract_all_metrics(symbol, data)
    all_metrics.append(metrics)
    print(f"✓ {symbol}")

# Convert to dataframe
metrics_df = pd.DataFrame(all_metrics)

# Save
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

print(f"\nTotal stocks    : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")

✓ TCS
✓ INFY
✓ HDFCBANK
✓ RELIANCE
✓ WIPRO
✓ PERSISTENT
✓ MPHASIS
✓ LTTS
✓ COFORGE
✓ HAPPSTMNDS
✓ CHOLAFIN
✓ MUTHOOTFIN
✓ MANAPPURAM
✓ IIFL
✓ CREDITACC
✓ GARFIBRES
✓ SUPRAJIT
✓ HIMATSEIDE
✓ IGPL
✓ HAL
✓ BEL
✓ BHEL
✓ MIDHANI
✓ PARAS
✓ VINATIORGA
✓ CLEAN
✓ FINEORG
✓ GALAXYSURF
✓ AAVAS
✓ SUNPHARMA
✓ DIVISLAB
✓ APLLTD
✓ GRANULES
✓ IPCALAB
✓ TITAN
✓ HAVELLS
✓ VGUARD
✓ SYMPHONY
✓ WONDERLA
✓ PIIND
✓ RALLIS
✓ DHANUKA
✓ DYNPRO
✓ GULFPETRO
✓ SUVIDHAA
✓ ABSLAMC


PermissionError: [Errno 13] Permission denied: '../data/fundamental_metrics.csv'

In [11]:
# ============================================
# SAFE DIVISION HELPER
# ============================================
def safe_divide(numerator, denominator):
    """Safe division — returns None if denominator is 0 or None"""
    try:
        if denominator and denominator > 0:
            return numerator / denominator
        return None
    except:
        return None


# ============================================
# SURVIVAL FILTER
# ============================================
def apply_survival_filter(df):
    """
    Eliminate stocks that are too dangerous to invest in.
    Returns dataframe with survival status and reason.
    """
    results = []

    for _, row in df.iterrows():
        symbol  = row['Symbol']
        reasons = []
        passed  = True

        # ── RULE 1: Promoter Holding < 30% ──
        if pd.notna(row.get('Promoter Holding')):
            if row['Promoter Holding'] < 30:
                passed = False
                reasons.append(
                    f"Low promoter holding: {row['Promoter Holding']}%"
                )

        # ── RULE 2: Debt to Equity > 2 ──
        # Exception: Financial companies naturally have high debt
        financial_keywords = ['Financial', 'Banking', 'Insurance', 'NBFC']
        is_financial       = any(
            kw in str(row.get('Sector', ''))
            for kw in financial_keywords
        )

        if not is_financial:
            if pd.notna(row.get('Debt to Equity')):
                if row['Debt to Equity'] > 2:
                    passed = False
                    reasons.append(
                        f"High debt/equity: {row['Debt to Equity']:.2f}"
                    )

        # ── RULE 3: Negative TTM Profit ──
        if pd.notna(row.get('TTM Profit')):
            if row['TTM Profit'] <= 0:
                passed = False
                reasons.append(
                    f"Negative TTM profit: {row['TTM Profit']}"
                )

        # ── RULE 4: Cash Flow Consistency ──
        cf_ratio = safe_divide(
            row.get('CF Positive Years'),
            row.get('CF Total Years')
        )
        if cf_ratio is not None:
            if cf_ratio < 0.7:
                passed = False
                reasons.append(
                    f"Poor CF consistency: "
                    f"{row.get('CF Positive Years')}/"
                    f"{row.get('CF Total Years')} years"
                )

        # ── RULE 5: Promoter Heavily Selling ──
        if pd.notna(row.get('Promoter Change 4Q')):
            if row['Promoter Change 4Q'] < -5:
                passed = False
                reasons.append(
                    f"Promoter selling heavily: "
                    f"{row['Promoter Change 4Q']}% in 4Q"
                )

        # ── RULE 6: Profit Positive Quarters ──
        profit_ratio = safe_divide(
            row.get('Profit Positive Q'),
            row.get('Total Quarters')
        )
        if profit_ratio is not None:
            if profit_ratio < 0.8:
                passed = False
                reasons.append(
                    f"Inconsistent profits: "
                    f"{row.get('Profit Positive Q')}/"
                    f"{row.get('Total Quarters')} quarters"
                )

        results.append({
            'Symbol'          : symbol,
            'Sector'          : row.get('Sector'),
            'Passed Survival' : passed,
            'Reason'          : ' | '.join(reasons) if reasons else 'All checks passed'
        })

    return pd.DataFrame(results)


# ============================================
# RUN SURVIVAL FILTER
# ============================================
survival_results = apply_survival_filter(metrics_df)

# Summary
passed = survival_results[survival_results['Passed Survival'] == True]
failed = survival_results[survival_results['Passed Survival'] == False]

print(f"Total stocks  : {len(survival_results)}")
print(f"Passed filter : {len(passed)}")
print(f"Failed filter : {len(failed)}")

print(f"\n{'='*60}")
print("FAILED STOCKS — Reasons:")
print(f"{'='*60}")
for _, row in failed.iterrows():
    print(f"{row['Symbol']:15} → {row['Reason']}")

print(f"\n{'='*60}")
print("PASSED STOCKS:")
print(f"{'='*60}")
print(passed['Symbol'].tolist())

Total stocks  : 46
Passed filter : 27
Failed filter : 19

FAILED STOCKS — Reasons:
INFY            → Low promoter holding: 14.52%
HDFCBANK        → Low promoter holding: 0.0% | Poor CF consistency: 7/12 years
MPHASIS         → Promoter selling heavily: -9.64% in 4Q
COFORGE         → Low promoter holding: 0.0%
CHOLAFIN        → Poor CF consistency: 0/12 years
MUTHOOTFIN      → Poor CF consistency: 1/11 years
MANAPPURAM      → Poor CF consistency: 1/12 years
IIFL            → Low promoter holding: 24.85% | Poor CF consistency: 3/12 years
CREDITACC       → Poor CF consistency: 1/7 years
IGPL            → Negative TTM profit: -7.0 | Inconsistent profits: 9/13 quarters
HAL             → Poor CF consistency: 6/9 years
BHEL            → Poor CF consistency: 8/12 years | Inconsistent profits: 9/13 quarters
PARAS           → Poor CF consistency: 4/6 years
CLEAN           → Promoter selling heavily: -24.01% in 4Q
AAVAS           → Poor CF consistency: 0/7 years
TITAN           → Poor CF consiste

In [12]:
# Check Titan cash flows
print("=== TITAN CASH FLOWS ===")
print(all_stock_data['TITAN']['Cash Flows'])

print("\n=== TITAN SECTOR ===")
print(all_stock_data['TITAN'].get('Sector Info'))

=== TITAN CASH FLOWS ===
                              Mar 2014  Mar 2015  Mar 2016  Mar 2017  \
Cash from Operating Activity    -555.0     503.0     576.0    1712.0   
Cash from Investing Activity    -271.0    -118.0    -159.0    -953.0   
Cash from Financing Activity     497.0   -1005.0    -505.0    -166.0   
Net Cash Flow                   -329.0    -620.0     -88.0     594.0   

                              Mar 2018  Mar 2019  Mar 2020  Mar 2021  \
Cash from Operating Activity     -51.0    1243.0    -348.0    4139.0   
Cash from Investing Activity      98.0    -797.0     235.0   -2799.0   
Cash from Financing Activity    -252.0    -489.0    -242.0   -1234.0   
Net Cash Flow                   -206.0     -43.0    -355.0     106.0   

                              Mar 2022  Mar 2023  Mar 2024  Mar 2025  
Cash from Operating Activity    -724.0    1370.0    1695.0    -541.0  
Cash from Investing Activity    1165.0   -1814.0    -189.0     546.0  
Cash from Financing Activity    -403.0  

In [13]:
# Check what data we actually have for Titan
print("=== TITAN CF DATA ===")
print(f"CF Positive Years : {metrics_df[metrics_df['Symbol']=='TITAN']['CF Positive Years'].values[0]}")
print(f"CF Total Years    : {metrics_df[metrics_df['Symbol']=='TITAN']['CF Total Years'].values[0]}")

print("\n=== TITAN QUARTERLY DATA ===")
print(f"Profit Positive Q : {metrics_df[metrics_df['Symbol']=='TITAN']['Profit Positive Q'].values[0]}")
print(f"Total Quarters    : {metrics_df[metrics_df['Symbol']=='TITAN']['Total Quarters'].values[0]}")

print("\n=== TCS QUARTERLY DATA ===")
print(f"Profit Positive Q : {metrics_df[metrics_df['Symbol']=='TCS']['Profit Positive Q'].values[0]}")
print(f"Total Quarters    : {metrics_df[metrics_df['Symbol']=='TCS']['Total Quarters'].values[0]}")

print("\n=== RAW QUARTERLY ROWS FOR TITAN ===")
titan_qr = all_stock_data['TITAN']['Quarterly Results']
print(f"Quarters available: {titan_qr.shape[1]}")
print(titan_qr.columns.tolist())

print("\n=== RAW ANNUAL CF ROWS FOR TITAN ===")
titan_cf = all_stock_data['TITAN']['Cash Flows']
print(f"Years available: {titan_cf.shape[1]}")
print(titan_cf.columns.tolist())

=== TITAN CF DATA ===
CF Positive Years : 7
CF Total Years    : 12

=== TITAN QUARTERLY DATA ===
Profit Positive Q : 13
Total Quarters    : 13

=== TCS QUARTERLY DATA ===
Profit Positive Q : 13
Total Quarters    : 13

=== RAW QUARTERLY ROWS FOR TITAN ===
Quarters available: 13
['Dec 2022', 'Mar 2023', 'Jun 2023', 'Sep 2023', 'Dec 2023', 'Mar 2024', 'Jun 2024', 'Sep 2024', 'Dec 2024', 'Mar 2025', 'Jun 2025', 'Sep 2025', 'Dec 2025']

=== RAW ANNUAL CF ROWS FOR TITAN ===
Years available: 12
['Mar 2014', 'Mar 2015', 'Mar 2016', 'Mar 2017', 'Mar 2018', 'Mar 2019', 'Mar 2020', 'Mar 2021', 'Mar 2022', 'Mar 2023', 'Mar 2024', 'Mar 2025']


In [14]:
# Check exactly what quarters we have
print("=== RELIANCE QUARTERS ===")
reliance_qr = all_stock_data['RELIANCE']['Quarterly Results']
print(f"Quarters: {reliance_qr.shape[1]}")
print(reliance_qr.columns.tolist())

print("\n=== RELIANCE ANNUAL ===")
reliance_pl = all_stock_data['RELIANCE']['Profit & Loss']
pl_clean = reliance_pl.drop(columns=['TTM'], errors='ignore')
print(f"Years: {pl_clean.shape[1]}")
print(pl_clean.columns.tolist())

print("\n=== HAPPSTMNDS QUARTERS ===")
happy_qr = all_stock_data['HAPPSTMNDS']['Quarterly Results']
print(f"Quarters: {happy_qr.shape[1]}")
print(happy_qr.columns.tolist())

=== RELIANCE QUARTERS ===
Quarters: 13
['Dec 2022', 'Mar 2023', 'Jun 2023', 'Sep 2023', 'Dec 2023', 'Mar 2024', 'Jun 2024', 'Sep 2024', 'Dec 2024', 'Mar 2025', 'Jun 2025', 'Sep 2025', 'Dec 2025']

=== RELIANCE ANNUAL ===
Years: 12
['Mar 2014', 'Mar 2015', 'Mar 2016', 'Mar 2017', 'Mar 2018', 'Mar 2019', 'Mar 2020', 'Mar 2021', 'Mar 2022', 'Mar 2023', 'Mar 2024', 'Mar 2025']

=== HAPPSTMNDS QUARTERS ===
Quarters: 13
['Dec 2022', 'Mar 2023', 'Jun 2023', 'Sep 2023', 'Dec 2023', 'Mar 2024', 'Jun 2024', 'Sep 2024', 'Dec 2024', 'Mar 2025', 'Jun 2025', 'Sep 2025', 'Dec 2025']


In [15]:
def safe_divide(numerator, denominator):
    try:
        if denominator and denominator > 0:
            return numerator / denominator
        return None
    except:
        return None


def get_cf_threshold(sector, industry):
    """Sector specific CF threshold"""
    sector   = str(sector)
    industry = str(industry)
    combined = sector + ' ' + industry

    # Skip CF rule — financial companies
    if any(s in combined for s in [
        'Financial', 'Banking', 'Insurance',
        'NBFC', 'Lending', 'Microfinance'
    ]):
        return None

    # Low threshold — working capital intensive
    if any(s in combined for s in [
        'Consumer', 'Retail', 'Jewellery', 'Gems',
        'Textile', 'FMCG', 'Watches', 'Durables',
        'Apparel', 'Fashion'
    ]):
        return 0.50

    # Low threshold — long project cycles
    if any(s in combined for s in [
        'Defence', 'Infrastructure', 'Construction',
        'Capital Goods', 'Engineering', 'Power'
    ]):
        return 0.60

    # Standard threshold
    return 0.70


def apply_survival_filter(df):
    results = []

    for _, row in df.iterrows():
        symbol   = row['Symbol']
        reasons  = []
        passed   = True
        sector   = str(row.get('Sector',   ''))
        industry = str(row.get('Industry', ''))

        # ── RULE 1: Promoter Holding < 30% ──
        if pd.notna(row.get('Promoter Holding')):
            if row['Promoter Holding'] < 30:
                passed = False
                reasons.append(
                    f"Low promoter holding: {row['Promoter Holding']}%"
                )

        # ── RULE 2: Debt to Equity > 2 ──
        financial_keywords = [
            'Financial', 'Banking', 'Insurance',
            'NBFC', 'Lending', 'Microfinance'
        ]
        is_financial = any(
            kw in sector or kw in industry
            for kw in financial_keywords
        )

        if not is_financial:
            if pd.notna(row.get('Debt to Equity')):
                if row['Debt to Equity'] > 2:
                    passed = False
                    reasons.append(
                        f"High debt/equity: {row['Debt to Equity']:.2f}"
                    )

        # ── RULE 3: Negative TTM Profit ──
        if pd.notna(row.get('TTM Profit')):
            if row['TTM Profit'] <= 0:
                passed = False
                reasons.append(
                    f"Negative TTM profit: {row['TTM Profit']}"
                )

        # ── RULE 4: Cash Flow Consistency ──
        cf_threshold = get_cf_threshold(sector, industry)

        if cf_threshold is not None:
            cf_ratio = safe_divide(
                row.get('CF Positive Years'),
                row.get('CF Total Years')
            )
            if cf_ratio is not None:
                if cf_ratio < cf_threshold:
                    passed = False
                    reasons.append(
                        f"Poor CF consistency: "
                        f"{row.get('CF Positive Years')}/"
                        f"{row.get('CF Total Years')} years "
                        f"(threshold: {cf_threshold:.0%})"
                    )

        # ── RULE 5: Promoter Heavily Selling ──
        if pd.notna(row.get('Promoter Change 4Q')):
            if row['Promoter Change 4Q'] < -5:
                passed = False
                reasons.append(
                    f"Promoter selling heavily: "
                    f"{row['Promoter Change 4Q']}% in 4Q"
                )

        # ── RULE 6: Profit Positive Quarters ──
        profit_ratio = safe_divide(
            row.get('Profit Positive Q'),
            row.get('Total Quarters')
        )
        if profit_ratio is not None:
            if profit_ratio < 0.8:
                passed = False
                reasons.append(
                    f"Inconsistent profits: "
                    f"{row.get('Profit Positive Q')}/"
                    f"{row.get('Total Quarters')} quarters"
                )

        results.append({
            'Symbol'          : symbol,
            'Sector'          : sector,
            'Industry'        : industry,
            'Passed Survival' : passed,
            'Reason'          : ' | '.join(reasons) if reasons else 'All checks passed'
        })

    return pd.DataFrame(results)


# Run updated filter
survival_results = apply_survival_filter(metrics_df)

passed = survival_results[survival_results['Passed Survival'] == True]
failed = survival_results[survival_results['Passed Survival'] == False]

print(f"Total stocks  : {len(survival_results)}")
print(f"Passed filter : {len(passed)}")
print(f"Failed filter : {len(failed)}")

print(f"\n{'='*60}")
print("FAILED STOCKS — Reasons:")
print(f"{'='*60}")
for _, row in failed.iterrows():
    print(f"{row['Symbol']:15} → {row['Reason']}")

print(f"\n{'='*60}")
print("PASSED STOCKS:")
print(f"{'='*60}")
for _, row in passed.iterrows():
    print(f"{row['Symbol']:15} {row['Sector']}")

Total stocks  : 46
Passed filter : 35
Failed filter : 11

FAILED STOCKS — Reasons:
INFY            → Low promoter holding: 14.52%
HDFCBANK        → Low promoter holding: 0.0%
MPHASIS         → Promoter selling heavily: -9.64% in 4Q
COFORGE         → Low promoter holding: 0.0%
IIFL            → Low promoter holding: 24.85%
IGPL            → Negative TTM profit: -7.0 | Inconsistent profits: 9/13 quarters
BHEL            → Inconsistent profits: 9/13 quarters
CLEAN           → Promoter selling heavily: -24.01% in 4Q
DYNPRO          → Low promoter holding: 29.42%
GULFPETRO       → Promoter selling heavily: -6.87% in 4Q
SUVIDHAA        → Negative TTM profit: -13.79 | Inconsistent profits: 1/13 quarters

PASSED STOCKS:
TCS             Information Technology
RELIANCE        Oil, Gas & Consumable Fuels
WIPRO           Information Technology
PERSISTENT      Information Technology
LTTS            Information Technology
HAPPSTMNDS      Information Technology
CHOLAFIN        Financial Services
MUTH

In [17]:
# Save all results
survival_results.to_csv('../data/survival_filter_results.csv', index=False)
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

passed_stocks = survival_results[
    survival_results['Passed Survival'] == True
]['Symbol'].tolist()

print(f"Stocks passing survival filter: {len(passed_stocks)}")
print(f"Saved survival_filter_results.csv")
print(f"Saved fundamental_metrics.csv")

Stocks passing survival filter: 35
Saved survival_filter_results.csv
Saved fundamental_metrics.csv


In [18]:
# Check which columns have missing values
print("=== MISSING VALUES PER METRIC ===")
missing = metrics_df.isnull().sum()
missing_pct = (missing / len(metrics_df) * 100).round(1)

missing_report = pd.DataFrame({
    'Missing Count' : missing,
    'Missing %'     : missing_pct
}).sort_values('Missing %', ascending=False)

# Show only columns with missing values
missing_report = missing_report[missing_report['Missing Count'] > 0]
print(missing_report.to_string())

=== MISSING VALUES PER METRIC ===
                         Missing Count  Missing %
Revenue CAGR 10Y                    19       41.3
Avg OPM 10Y                         17       37.0
Profit CAGR 10Y                     14       30.4
Avg OPM 5Y                           9       19.6
Revenue CAGR 5Y                      9       19.6
Revenue Accelerating                 9       19.6
Revenue YoY Q                        9       19.6
TTM Revenue                          9       19.6
Debt Reducing                        9       19.6
Latest Debt                          9       19.6
Avg ROCE 5Y                          9       19.6
Latest ROCE                          9       19.6
Debt to Equity                       9       19.6
ROCE Improving                       9       19.6
Margin Improving                     9       19.6
Avg OPM 4Q                           9       19.6
Latest OPM Q                         9       19.6
Revenue Consecutive YoY              7       15.2
Profit YoY Q    

In [19]:
# Find which stocks have the most missing values
print("=== MISSING VALUES PER STOCK ===")
stock_missing = metrics_df.isnull().sum(axis=1)
stock_missing.index = metrics_df['Symbol']
stock_missing = stock_missing.sort_values(ascending=False)
print(stock_missing[stock_missing > 0])

=== MISSING VALUES PER STOCK ===
Symbol
GULFPETRO     26
WONDERLA      26
CREDITACC     20
AAVAS         18
HDFCBANK      17
IIFL          17
CHOLAFIN      17
MUTHOOTFIN    17
MANAPPURAM    17
SUVIDHAA       8
VINATIORGA     3
MIDHANI        3
PARAS          3
HAPPSTMNDS     3
VGUARD         3
CLEAN          3
ABSLAMC        3
SYMPHONY       2
FINEORG        2
IGPL           2
BHEL           1
dtype: int64


In [20]:
# Let's check each group

# Group 1 — GULFPETRO, WONDERLA (26 missing)
print("=== GULFPETRO SECTIONS AVAILABLE ===")
print(list(all_stock_data['GULFPETRO'].keys()))

print("\n=== WONDERLA SECTIONS AVAILABLE ===")
print(list(all_stock_data['WONDERLA'].keys()))

# Group 2 — Financial companies (17-20 missing)
print("\n=== CHOLAFIN SECTIONS AVAILABLE ===")
print(list(all_stock_data['CHOLAFIN'].keys()))
print("\n=== CHOLAFIN RATIOS ===")
print(all_stock_data['CHOLAFIN'].get('Ratios'))

# Group 3 — Small missing (2-3 missing)
print("\n=== HAPPSTMNDS PROFIT & LOSS ===")
pl = all_stock_data['HAPPSTMNDS']['Profit & Loss']
pl_clean = pl.drop(columns=['TTM'], errors='ignore')
print(f"Years available: {pl_clean.shape[1]}")
print(pl_clean.columns.tolist())

=== GULFPETRO SECTIONS AVAILABLE ===
['Quarterly Results', 'Profit & Loss', 'Balance Sheet', 'Cash Flows', 'Ratios', 'Shareholding Pattern', 'Sector Info']

=== WONDERLA SECTIONS AVAILABLE ===
['Quarterly Results', 'Profit & Loss', 'Balance Sheet', 'Cash Flows', 'Ratios', 'Shareholding Pattern', 'Sector Info']

=== CHOLAFIN SECTIONS AVAILABLE ===
['Quarterly Results', 'Profit & Loss', 'Balance Sheet', 'Cash Flows', 'Ratios', 'Shareholding Pattern', 'Sector Info']

=== CHOLAFIN RATIOS ===
       Mar 2014  Mar 2015  Mar 2016  Mar 2017  Mar 2018  Mar 2019  Mar 2020  \
ROE %      17.0      18.0      18.0      18.0      20.0      21.0      15.0   

       Mar 2021  Mar 2022  Mar 2023  Mar 2024  Mar 2025  
ROE %      17.0      20.0      20.0      20.0      20.0  

=== HAPPSTMNDS PROFIT & LOSS ===
Years available: 7
['Mar 2019', 'Mar 2020', 'Mar 2021', 'Mar 2022', 'Mar 2023', 'Mar 2024', 'Mar 2025']


In [21]:
def extract_all_metrics(symbol, data):
    """
    Extract all fundamental metrics with zero redundancy
    Annual data for long term trends
    Quarterly data for recent momentum
    Handles missing data gracefully
    """
    m = {'Symbol': symbol}

    try:
        # ════════════════════════════════════════
        # SECTOR INFO
        # ════════════════════════════════════════
        sector_info         = data.get('Sector Info', {})
        m['Sector']         = sector_info.get('Sector',       None)
        m['Industry']       = sector_info.get('Industry',     None)
        m['Broad Sector']   = sector_info.get('Broad Sector', None)

        # ════════════════════════════════════════
        # ANNUAL P&L → Long term CAGR only
        # ════════════════════════════════════════
        pl = data.get('Profit & Loss')
        if pl is not None:
            pl_clean = pl.drop(columns=['TTM'], errors='ignore')

            # Revenue CAGR — dynamic lookback
            if 'Sales' in pl_clean.index:
                sales = pl_clean.loc['Sales'].dropna()

                # 5 year CAGR
                if len(sales) >= 6:
                    m['Revenue CAGR 5Y'] = round(
                        ((sales.iloc[-1] / sales.iloc[-6]) ** (1/5) - 1) * 100, 2
                    )

                # 10 year CAGR
                if len(sales) >= 11:
                    m['Revenue CAGR 10Y'] = round(
                        ((sales.iloc[-1] / sales.iloc[-11]) ** (1/10) - 1) * 100, 2
                    )

                # Dynamic CAGR — use whatever history is available
                if len(sales) >= 3:
                    years_available      = min(len(sales) - 1, 10)
                    base                 = sales.iloc[-years_available - 1]
                    if base > 0:
                        m['Revenue CAGR Max'] = round(
                            ((sales.iloc[-1] / base) ** (1/years_available) - 1) * 100, 2
                        )
                        m['Revenue CAGR Years'] = years_available

                # Long term OPM
                if 'OPM %' in pl_clean.index:
                    opm              = pl_clean.loc['OPM %'].dropna()
                    m['Avg OPM 5Y']  = round(opm.iloc[-5:].mean(),  2) if len(opm) >= 5  else round(opm.mean(), 2) if len(opm) > 0 else None
                    m['Avg OPM 10Y'] = round(opm.iloc[-10:].mean(), 2) if len(opm) >= 10 else None

            # Profit CAGR — dynamic lookback
            if 'Net Profit' in pl_clean.index:
                profit = pl_clean.loc['Net Profit'].dropna()

                if len(profit) >= 6 and profit.iloc[-6] > 0:
                    m['Profit CAGR 5Y']  = round(
                        ((profit.iloc[-1] / profit.iloc[-6]) ** (1/5) - 1) * 100, 2
                    )

                if len(profit) >= 11 and profit.iloc[-11] > 0:
                    m['Profit CAGR 10Y'] = round(
                        ((profit.iloc[-1] / profit.iloc[-11]) ** (1/10) - 1) * 100, 2
                    )

                # Dynamic profit CAGR
                if len(profit) >= 3:
                    years_available = min(len(profit) - 1, 10)
                    base            = profit.iloc[-years_available - 1]
                    if base > 0:
                        m['Profit CAGR Max']  = round(
                            ((profit.iloc[-1] / base) ** (1/years_available) - 1) * 100, 2
                        )
                        m['Profit CAGR Years'] = years_available

        # ════════════════════════════════════════
        # QUARTERLY DATA → Recent momentum
        # ════════════════════════════════════════
        qr = data.get('Quarterly Results')
        if qr is not None:
            qr_clean = qr.drop(columns=['TTM'], errors='ignore')

            # Revenue recent momentum
            if 'Sales' in qr_clean.index:
                sales_q = qr_clean.loc['Sales'].dropna()

                # TTM Revenue (last 4 quarters)
                if len(sales_q) >= 4:
                    m['TTM Revenue']             = round(sales_q.iloc[-4:].sum(), 2)

                # YoY growth latest quarter
                if len(sales_q) >= 5:
                    latest                       = sales_q.iloc[-1]
                    year_ago                     = sales_q.iloc[-5]
                    if year_ago > 0:
                        m['Revenue YoY Q']       = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )

                # Count consecutive YoY improvements
                yoy_improvements = 0
                for i in range(1, min(5, len(sales_q) - 4)):
                    curr = sales_q.iloc[-i]
                    prev = sales_q.iloc[-i - 4]
                    if prev > 0 and curr > prev:
                        yoy_improvements += 1
                    else:
                        break
                m['Revenue Consecutive YoY']     = yoy_improvements

                # Revenue acceleration
                if len(sales_q) >= 9:
                    recent_yoy                   = (sales_q.iloc[-1] - sales_q.iloc[-5]) / sales_q.iloc[-5] * 100
                    previous_yoy                 = (sales_q.iloc[-5] - sales_q.iloc[-9]) / sales_q.iloc[-9] * 100
                    m['Revenue Accelerating']    = recent_yoy > previous_yoy

            # Profit recent momentum
            if 'Net Profit' in qr_clean.index:
                profit_q = qr_clean.loc['Net Profit'].dropna()

                # TTM Profit
                if len(profit_q) >= 4:
                    m['TTM Profit']              = round(profit_q.iloc[-4:].sum(), 2)

                # YoY profit growth
                if len(profit_q) >= 5:
                    latest                       = profit_q.iloc[-1]
                    year_ago                     = profit_q.iloc[-5]
                    if year_ago > 0:
                        m['Profit YoY Q']        = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )

                # Profit positive quarters
                m['Profit Positive Q']           = int((profit_q > 0).sum())
                m['Total Quarters']              = len(profit_q)

            # Margin recent trend
            if 'OPM %' in qr_clean.index:
                opm_q = qr_clean.loc['OPM %'].dropna()

                m['Latest OPM Q']                = opm_q.iloc[-1]  if len(opm_q) > 0 else None
                m['Avg OPM 4Q']                  = round(opm_q.iloc[-4:].mean(), 2) if len(opm_q) >= 4 else None

                # Margin improving vs year ago
                if len(opm_q) >= 5:
                    m['Margin Improving']        = bool(opm_q.iloc[-1] >= opm_q.iloc[-5])

            # EPS recent trend
            if 'EPS in Rs' in qr_clean.index:
                eps_q = qr_clean.loc['EPS in Rs'].dropna()

                m['Latest EPS Q']                = eps_q.iloc[-1] if len(eps_q) > 0 else None

                if len(eps_q) >= 5:
                    latest                       = eps_q.iloc[-1]
                    year_ago                     = eps_q.iloc[-5]
                    if year_ago > 0:
                        m['EPS YoY Growth']      = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )

        # ════════════════════════════════════════
        # BALANCE SHEET → Financial structure
        # ════════════════════════════════════════
        bs = data.get('Balance Sheet')
        if bs is not None:
            if 'Borrowings' in bs.index:
                debt                             = bs.loc['Borrowings'].dropna()
                m['Latest Debt']                 = debt.iloc[-1]  if len(debt) > 0 else None

                # Debt reducing over 3 years?
                if len(debt) >= 4:
                    m['Debt Reducing']           = bool(debt.iloc[-1] < debt.iloc[-4])

            if 'Reserves' in bs.index and 'Equity Capital' in bs.index:
                reserves                         = bs.loc['Reserves'].dropna()
                eq_cap                           = bs.loc['Equity Capital'].dropna()

                if len(reserves) > 0 and len(eq_cap) > 0:
                    equity                       = reserves.iloc[-1] + eq_cap.iloc[-1]
                    m['Latest Equity']           = round(equity, 2)

                    # ROE from balance sheet
                    if m.get('TTM Profit') and equity > 0:
                        m['ROE']                 = round(
                            m['TTM Profit'] / equity * 100, 2
                        )

                    # Debt to Equity
                    if m.get('Latest Debt') is not None and equity > 0:
                        m['Debt to Equity']      = round(
                            m['Latest Debt'] / equity, 2
                        )

        # ════════════════════════════════════════
        # CASH FLOWS → Cash quality
        # ════════════════════════════════════════
        cf = data.get('Cash Flows')
        if cf is not None:
            if 'Cash from Operating Activity' in cf.index:
                op_cf                            = cf.loc['Cash from Operating Activity'].dropna()
                m['Latest Operating CF']         = op_cf.iloc[-1] if len(op_cf) > 0 else None
                m['CF Positive Years']           = int((op_cf > 0).sum())
                m['CF Total Years']              = len(op_cf)

                # CF growing?
                if len(op_cf) >= 4:
                    m['CF Growing']              = bool(op_cf.iloc[-1] > op_cf.iloc[-4])

        # ════════════════════════════════════════
        # RATIOS → Efficiency
        # ════════════════════════════════════════
        ratios = data.get('Ratios')
        if ratios is not None:

            # ROCE — for non financial companies
            if 'ROCE %' in ratios.index:
                roce                             = ratios.loc['ROCE %'].dropna()
                m['Latest ROCE']                 = roce.iloc[-1] if len(roce) > 0 else None

                if len(roce) >= 4:
                    m['ROCE Improving']          = bool(roce.iloc[-1] > roce.iloc[-4])

                if len(roce) >= 5:
                    m['Avg ROCE 5Y']             = round(roce.iloc[-5:].mean(), 2)

            # ROE from Ratios — for financial companies
            # More accurate than calculating from P&L
            if 'ROE %' in ratios.index:
                roe_series                       = ratios.loc['ROE %'].dropna()
                m['ROE from Ratios']             = roe_series.iloc[-1] if len(roe_series) > 0 else None
                m['ROE Avg 5Y']                  = round(roe_series.iloc[-5:].mean(), 2) if len(roe_series) >= 5 else None
                m['ROE Improving']               = bool(roe_series.iloc[-1] > roe_series.iloc[-4]) if len(roe_series) >= 4 else None

        # ════════════════════════════════════════
        # SHAREHOLDING → Ownership signals
        # ════════════════════════════════════════
        sh = data.get('Shareholding Pattern')
        if sh is not None:
            if 'Promoters' in sh.index:
                promoter                         = sh.loc['Promoters'].dropna()
                m['Promoter Holding']            = promoter.iloc[-1] if len(promoter) > 0 else None

                # Change over last 4 quarters
                if len(promoter) >= 5:
                    m['Promoter Change 4Q']      = round(
                        promoter.iloc[-1] - promoter.iloc[-5], 2
                    )

                # Change over last 8 quarters
                if len(promoter) >= 9:
                    m['Promoter Change 8Q']      = round(
                        promoter.iloc[-1] - promoter.iloc[-9], 2
                    )

            if 'FIIs' in sh.index:
                fii                              = sh.loc['FIIs'].dropna()
                m['FII Holding']                 = fii.iloc[-1] if len(fii) > 0 else None
                if len(fii) >= 5:
                    m['FII Change 4Q']           = round(fii.iloc[-1] - fii.iloc[-5], 2)

            if 'DIIs' in sh.index:
                dii                              = sh.loc['DIIs'].dropna()
                m['DII Holding']                 = dii.iloc[-1] if len(dii) > 0 else None
                if len(dii) >= 5:
                    m['DII Change 4Q']           = round(dii.iloc[-1] - dii.iloc[-5], 2)

    except Exception as e:
        print(f"Error extracting {symbol}: {e}")

    return m


# ════════════════════════════════════════════
# RE-RUN FOR ALL 46 STOCKS
# ════════════════════════════════════════════
all_metrics = []

for symbol, data in all_stock_data.items():
    metrics = extract_all_metrics(symbol, data)
    all_metrics.append(metrics)
    print(f"✓ {symbol}")

metrics_df = pd.DataFrame(all_metrics)

# Save
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

print(f"\nTotal stocks     : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")

# Check missing values
print(f"\n=== MISSING VALUES AFTER FIX ===")
missing     = metrics_df.isnull().sum()
missing_pct = (missing / len(metrics_df) * 100).round(1)

missing_report = pd.DataFrame({
    'Missing Count' : missing,
    'Missing %'     : missing_pct
}).sort_values('Missing %', ascending=False)

missing_report = missing_report[missing_report['Missing Count'] > 0]
print(missing_report.to_string())

✓ TCS
✓ INFY
✓ HDFCBANK
✓ RELIANCE
✓ WIPRO
✓ PERSISTENT
✓ MPHASIS
✓ LTTS
✓ COFORGE
✓ HAPPSTMNDS
✓ CHOLAFIN
✓ MUTHOOTFIN
✓ MANAPPURAM
✓ IIFL
✓ CREDITACC
✓ GARFIBRES
✓ SUPRAJIT
✓ HIMATSEIDE
✓ IGPL
✓ HAL
✓ BEL
✓ BHEL
✓ MIDHANI
✓ PARAS
✓ VINATIORGA
✓ CLEAN
✓ FINEORG
✓ GALAXYSURF
✓ AAVAS
✓ SUNPHARMA
✓ DIVISLAB
✓ APLLTD
✓ GRANULES
✓ IPCALAB
✓ TITAN
✓ HAVELLS
✓ VGUARD
✓ SYMPHONY
✓ WONDERLA
✓ PIIND
✓ RALLIS
✓ DHANUKA
✓ DYNPRO
✓ GULFPETRO
✓ SUVIDHAA
✓ ABSLAMC

Total stocks     : 46
Metrics per stock: 49

=== MISSING VALUES AFTER FIX ===
                         Missing Count  Missing %
ROE from Ratios                     39       84.8
ROE Avg 5Y                          39       84.8
ROE Improving                       39       84.8
Revenue CAGR 10Y                    19       41.3
Avg OPM 10Y                         17       37.0
Profit CAGR 10Y                     14       30.4
Avg OPM 5Y                           9       19.6
TTM Revenue                          9       19.6
Revenue CAGR 5Y 

In [22]:
# Create unified ROE column
# For financial companies → use ROE from Ratios (more accurate)
# For others → use ROE calculated from P&L/Balance sheet

def get_final_roe(row):
    # If ROE from Ratios available → use it
    if pd.notna(row.get('ROE from Ratios')):
        return row['ROE from Ratios']
    # Otherwise use calculated ROE
    elif pd.notna(row.get('ROE')):
        return row['ROE']
    return None

metrics_df['Final ROE'] = metrics_df.apply(get_final_roe, axis=1)

# Check result
print("=== FINAL ROE — Sample ===")
print(metrics_df[['Symbol', 'Sector', 'ROE', 
                   'ROE from Ratios', 'Final ROE']].to_string())

=== FINAL ROE — Sample ===
        Symbol                          Sector    ROE  ROE from Ratios  Final ROE
0          TCS          Information Technology  45.07              NaN      45.07
1         INFY          Information Technology  27.10              NaN      27.10
2     HDFCBANK              Financial Services  14.36             14.0      14.00
3     RELIANCE     Oil, Gas & Consumable Fuels  11.15              NaN      11.15
4        WIPRO          Information Technology  15.58              NaN      15.58
5   PERSISTENT          Information Technology  24.22              NaN      24.22
6      MPHASIS          Information Technology  18.70              NaN      18.70
7         LTTS          Information Technology  20.33              NaN      20.33
8      COFORGE          Information Technology  20.09              NaN      20.09
9   HAPPSTMNDS          Information Technology  11.36              NaN      11.36
10    CHOLAFIN              Financial Services  18.69             20.0 

In [23]:
# Save updated metrics
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

# Re-run survival filter with updated metrics
survival_results = apply_survival_filter(metrics_df)
survival_results.to_csv('../data/survival_filter_results.csv', index=False)

passed = survival_results[survival_results['Passed Survival'] == True]
failed = survival_results[survival_results['Passed Survival'] == False]

print(f"Total stocks  : {len(survival_results)}")
print(f"Passed filter : {len(passed)}")
print(f"Failed filter : {len(failed)}")
print(f"Metrics saved : {len(metrics_df.columns)} columns")
print(f"All files saved to data folder ✅")

Total stocks  : 46
Passed filter : 35
Failed filter : 11
Metrics saved : 50 columns
All files saved to data folder ✅


In [24]:
# Check Wonderla in detail
print("=== WONDERLA SECTIONS ===")
print(list(all_stock_data['WONDERLA'].keys()))

print("\n=== WONDERLA PROFIT & LOSS ===")
wonderla_pl = all_stock_data['WONDERLA']['Profit & Loss']
print(wonderla_pl)

print("\n=== WONDERLA QUARTERLY ===")
wonderla_qr = all_stock_data['WONDERLA']['Quarterly Results']
print(wonderla_qr)

print("\n=== WONDERLA METRICS ROW ===")
wonderla_metrics = metrics_df[metrics_df['Symbol'] == 'WONDERLA']
# Show only non-null values
for col in wonderla_metrics.columns:
    val = wonderla_metrics[col].values[0]
    if pd.notna(val):
        print(f"  {col:30} : {val}")

print("\n=== WONDERLA NULL VALUES ===")
null_cols = wonderla_metrics.columns[wonderla_metrics.isnull().any()].tolist()
print(f"Missing {len(null_cols)} metrics:")
print(null_cols)

=== WONDERLA SECTIONS ===
['Quarterly Results', 'Profit & Loss', 'Balance Sheet', 'Cash Flows', 'Ratios', 'Shareholding Pattern', 'Sector Info']

=== WONDERLA PROFIT & LOSS ===
Empty DataFrame
Columns: []
Index: [Sales, Expenses, Operating Profit, OPM %, Other Income, Interest, Depreciation, Profit before tax, Tax %, Net Profit, EPS in Rs, Dividend Payout %]

=== WONDERLA QUARTERLY ===
Empty DataFrame
Columns: []
Index: [Sales, Expenses, Operating Profit, OPM %, Other Income, Interest, Depreciation, Profit before tax, Tax %, Net Profit, EPS in Rs]

=== WONDERLA METRICS ROW ===
  Symbol                         : WONDERLA
  Sector                         : Consumer Services
  Industry                       : Amusement Parks/ Other Recreation
  Broad Sector                   : Consumer Discretionary
  Revenue Consecutive YoY        : 0.0
  Profit Positive Q              : 0
  Total Quarters                 : 0
  CF Positive Years              : 0
  CF Total Years                 : 0
  Pro

In [25]:
# Check raw HTML table for Wonderla
url     = "https://www.screener.in/company/WONDERLA/consolidated/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers, timeout=15)
soup     = BeautifulSoup(response.text, 'html.parser')

# Find P&L section
for section in soup.find_all('section'):
    heading = section.find('h2')
    if heading and 'Profit' in heading.text:
        table = section.find('table')
        if table:
            # Check thead
            thead = table.find('thead')
            print("THEAD found:", thead is not None)
            if thead:
                print("THEAD content:")
                print(thead)
            else:
                print("NO THEAD — checking first row:")
                first_row = table.find('tr')
                print(first_row)
        break

THEAD found: True
THEAD content:
<thead>
<tr>
<th class="text"></th>
</tr>
</thead>


In [26]:
# Try standalone page for Wonderla
url      = "https://www.screener.in/company/WONDERLA/"
headers  = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers, timeout=15)
soup     = BeautifulSoup(response.text, 'html.parser')

# Find P&L section
for section in soup.find_all('section'):
    heading = section.find('h2')
    if heading and 'Profit' in heading.text:
        table = section.find('table')
        if table:
            thead = table.find('thead')
            print("THEAD found:", thead is not None)
            if thead:
                ths = thead.find_all('th')
                print(f"Columns found: {len(ths)}")
                for th in ths:
                    print(f"  {th.get_text(strip=True)}")
        break

THEAD found: True
Columns found: 14
  
  Mar 2014
  Mar 2015
  Mar 2016
  Mar 2017
  Mar 2018
  Mar 2019
  Mar 2020
  Mar 2021
  Mar 2022
  Mar 2023
  Mar 2024
  Mar 2025
  TTM


In [27]:
# ============================================
# UPDATED SCRAPER — Auto fallback to standalone
# ============================================

def extract_table(table):
    try:
        thead   = table.find('thead')
        columns = []
        if thead:
            for th in thead.find_all('th'):
                text = th.get_text(strip=True)
                columns.append(text if text else 'Metric')

        tbody = table.find('tbody')
        if not tbody:
            return None

        rows = tbody.find_all('tr')
        data = {}

        for row in rows:
            cells       = row.find_all('td')
            if not cells:
                continue
            metric_name = cells[0].get_text(strip=True)
            metric_name = metric_name.replace('+', '').strip()

            skip_keywords = ['Raw PDF', 'PDF', 'Source']
            if any(kw in metric_name for kw in skip_keywords):
                continue

            values = []
            for cell in cells[1:]:
                val = cell.get_text(strip=True)
                val = val.replace(',', '').replace('%', '').strip()
                try:
                    values.append(float(val))
                except:
                    values.append(val if val else None)

            data[metric_name] = values

        if not data:
            return None

        col_names = columns[1:] if len(columns) > 1 else list(range(len(values)))
        df        = pd.DataFrame(data, index=col_names).T
        return df

    except Exception as e:
        print(f"Table extraction error: {e}")
        return None


def scrape_screener_complete(symbol):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

        # Try consolidated first, then standalone
        urls = [
            f"https://www.screener.in/company/{symbol}/consolidated/",
            f"https://www.screener.in/company/{symbol}/",
        ]

        soup = None
        for url in urls:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code != 200:
                continue

            temp_soup = BeautifulSoup(response.text, 'html.parser')

            # Check if P&L table has actual columns
            has_data  = False
            for section in temp_soup.find_all('section'):
                heading = section.find('h2')
                if heading and 'Profit' in heading.text:
                    table = section.find('table')
                    if table:
                        thead = table.find('thead')
                        if thead:
                            ths = thead.find_all('th')
                            # More than 1 th means real data exists
                            if len(ths) > 1:
                                has_data = True
                    break

            if has_data:
                soup = temp_soup
                break

        if soup is None:
            print(f"No data found for {symbol}")
            return None

        # Extract all sections
        target_sections = [
            'Quarterly Results',
            'Profit & Loss',
            'Balance Sheet',
            'Cash Flows',
            'Ratios',
            'Shareholding Pattern',
        ]

        result = {}

        for section in soup.find_all('section'):
            heading = section.find('h2')
            if not heading:
                continue
            section_name = heading.text.strip()
            if section_name not in target_sections:
                continue
            table = section.find('table')
            if not table:
                continue
            df = extract_table(table)
            if df is not None:
                result[section_name] = df

        # Extract sector info
        for section in soup.find_all('section'):
            heading = section.find('h2')
            if heading and 'Peer' in heading.text:
                links       = section.find_all('a')
                sector_info = {}
                titles      = ['Broad Sector', 'Sector',
                               'Broad Industry', 'Industry']
                for link in links:
                    title = link.get('title', '')
                    if title in titles:
                        sector_info[title] = link.text.strip()
                result['Sector Info'] = sector_info
                break

        return result if result else None

    except Exception as e:
        print(f"Error scraping {symbol}: {e}")
        return None


# ============================================
# RE-FETCH STOCKS WITH EMPTY DATA
# ============================================
refetch_stocks = ['WONDERLA', 'GULFPETRO', 'CREDITACC', 'AAVAS']

print("Re-fetching with updated scraper...")
for symbol in refetch_stocks:
    print(f"\nFetching {symbol}...", end=" ")
    data = scrape_screener_complete(symbol)
    if data:
        all_stock_data[symbol] = data
        sections = [k for k in data.keys() if k != 'Sector Info']
        print(f"✓ ({len(sections)} sections)")

        # Check P&L data
        pl = data.get('Profit & Loss')
        if pl is not None:
            print(f"  P&L columns : {pl.shape[1]}")
            print(f"  P&L rows    : {pl.shape[0]}")
        else:
            print("  P&L         : None")

        # Check Quarterly data
        qr = data.get('Quarterly Results')
        if qr is not None:
            print(f"  QR columns  : {qr.shape[1]}")
        else:
            print("  QR          : None")
    else:
        print(f"✗ Failed")
    time.sleep(2)

Re-fetching with updated scraper...

Fetching WONDERLA... ✓ (6 sections)
  P&L columns : 13
  P&L rows    : 12
  QR columns  : 13

Fetching GULFPETRO... ✓ (6 sections)
  P&L columns : 13
  P&L rows    : 12
  QR columns  : 13

Fetching CREDITACC... ✓ (6 sections)
  P&L columns : 8
  P&L rows    : 12
  QR columns  : 13

Fetching AAVAS... ✓ (6 sections)
  P&L columns : 7
  P&L rows    : 12
  QR columns  : 13


In [28]:
# Re-extract metrics for all 46 stocks
all_metrics = []

for symbol, data in all_stock_data.items():
    metrics = extract_all_metrics(symbol, data)
    all_metrics.append(metrics)
    print(f"✓ {symbol}")

metrics_df = pd.DataFrame(all_metrics)

# Add unified Final ROE
def get_final_roe(row):
    if pd.notna(row.get('ROE from Ratios')):
        return row['ROE from Ratios']
    elif pd.notna(row.get('ROE')):
        return row['ROE']
    return None

metrics_df['Final ROE'] = metrics_df.apply(get_final_roe, axis=1)

# Save
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

print(f"\nTotal stocks     : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")

# Check missing values
print(f"\n=== MISSING VALUES AFTER FIX ===")
missing     = metrics_df.isnull().sum()
missing_pct = (missing / len(metrics_df) * 100).round(1)

missing_report = pd.DataFrame({
    'Missing Count' : missing,
    'Missing %'     : missing_pct
}).sort_values('Missing %', ascending=False)

missing_report = missing_report[missing_report['Missing Count'] > 0]
print(missing_report.to_string())

✓ TCS
✓ INFY
✓ HDFCBANK
✓ RELIANCE
✓ WIPRO
✓ PERSISTENT
✓ MPHASIS
✓ LTTS
✓ COFORGE
✓ HAPPSTMNDS
✓ CHOLAFIN
✓ MUTHOOTFIN
✓ MANAPPURAM
✓ IIFL
✓ CREDITACC
✓ GARFIBRES
✓ SUPRAJIT
✓ HIMATSEIDE
✓ IGPL
✓ HAL
✓ BEL
✓ BHEL
✓ MIDHANI
✓ PARAS
✓ VINATIORGA
✓ CLEAN
✓ FINEORG
✓ GALAXYSURF
✓ AAVAS
✓ SUNPHARMA
✓ DIVISLAB
✓ APLLTD
✓ GRANULES
✓ IPCALAB
✓ TITAN
✓ HAVELLS
✓ VGUARD
✓ SYMPHONY
✓ WONDERLA
✓ PIIND
✓ RALLIS
✓ DHANUKA
✓ DYNPRO
✓ GULFPETRO
✓ SUVIDHAA
✓ ABSLAMC

Total stocks     : 46
Metrics per stock: 50

=== MISSING VALUES AFTER FIX ===
                         Missing Count  Missing %
ROE Avg 5Y                          39       84.8
ROE from Ratios                     39       84.8
ROE Improving                       39       84.8
Revenue CAGR 10Y                    17       37.0
Avg OPM 10Y                         15       32.6
Profit CAGR 10Y                     12       26.1
Revenue CAGR Years                   7       15.2
TTM Revenue                          7       15.2
Revenue Accelera

In [29]:
# Save final metrics and survival filter
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

# Re-run survival filter
survival_results = apply_survival_filter(metrics_df)
survival_results.to_csv('../data/survival_filter_results.csv', index=False)

# Save updated raw data
import pickle
with open('../data/raw_stock_data.pkl', 'wb') as f:
    pickle.dump(all_stock_data, f)

passed = survival_results[survival_results['Passed Survival'] == True]
failed = survival_results[survival_results['Passed Survival'] == False]

print(f"Total stocks     : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")
print(f"Passed survival  : {len(passed)}")
print(f"Failed survival  : {len(failed)}")
print(f"\nAll files saved ✅")

Total stocks     : 46
Metrics per stock: 50
Passed survival  : 35
Failed survival  : 11

All files saved ✅


In [30]:
# Check actual row names for financial companies
print("=== CHOLAFIN P&L ROW NAMES ===")
print(all_stock_data['CHOLAFIN']['Profit & Loss'].index.tolist())

print("\n=== CHOLAFIN QUARTERLY ROW NAMES ===")
print(all_stock_data['CHOLAFIN']['Quarterly Results'].index.tolist())

print("\n=== CHOLAFIN BALANCE SHEET ROW NAMES ===")
print(all_stock_data['CHOLAFIN']['Balance Sheet'].index.tolist())

print("\n=== TCS P&L ROW NAMES (for comparison) ===")
print(all_stock_data['TCS']['Profit & Loss'].index.tolist())

=== CHOLAFIN P&L ROW NAMES ===
['Revenue', 'Interest', 'Expenses', 'Financing Profit', 'Financing Margin %', 'Other Income', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit', 'EPS in Rs', 'Dividend Payout %']

=== CHOLAFIN QUARTERLY ROW NAMES ===
['Revenue', 'Interest', 'Expenses', 'Financing Profit', 'Financing Margin %', 'Other Income', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit', 'EPS in Rs', 'Gross NPA %', 'Net NPA %']

=== CHOLAFIN BALANCE SHEET ROW NAMES ===
['Equity Capital', 'Reserves', 'Borrowing', 'Other Liabilities', 'Total Liabilities', 'Fixed Assets', 'CWIP', 'Investments', 'Other Assets', 'Total Assets']

=== TCS P&L ROW NAMES (for comparison) ===
['Sales', 'Expenses', 'Operating Profit', 'OPM %', 'Other Income', 'Interest', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit', 'EPS in Rs', 'Dividend Payout %']


In [31]:
def extract_all_metrics(symbol, data):
    """
    Extract all fundamental metrics with zero redundancy
    Handles both regular and financial company row names
    """
    m = {'Symbol': symbol}

    try:
        # ════════════════════════════════════════
        # SECTOR INFO
        # ════════════════════════════════════════
        sector_info       = data.get('Sector Info', {})
        m['Sector']       = sector_info.get('Sector',       None)
        m['Industry']     = sector_info.get('Industry',     None)
        m['Broad Sector'] = sector_info.get('Broad Sector', None)

        # Detect if financial company
        financial_keywords = [
            'Financial', 'Banking', 'Insurance',
            'NBFC', 'Lending', 'Microfinance'
        ]
        is_financial = any(
            kw in str(m.get('Sector', ''))
            for kw in financial_keywords
        )

        # ════════════════════════════════════════
        # ROW NAME MAPPING
        # Financial companies use different names
        # ════════════════════════════════════════
        if is_financial:
            revenue_row = 'Revenue'
            margin_row  = 'Financing Margin %'
            profit_row  = 'Financing Profit'
            debt_row    = 'Borrowing'
        else:
            revenue_row = 'Sales'
            margin_row  = 'OPM %'
            profit_row  = 'Operating Profit'
            debt_row    = 'Borrowings'

        # ════════════════════════════════════════
        # ANNUAL P&L → Long term CAGR
        # ════════════════════════════════════════
        pl = data.get('Profit & Loss')
        if pl is not None:
            pl_clean = pl.drop(columns=['TTM'], errors='ignore')

            # Revenue CAGR
            if revenue_row in pl_clean.index:
                sales = pl_clean.loc[revenue_row].dropna()

                if len(sales) >= 6:
                    m['Revenue CAGR 5Y']  = round(
                        ((sales.iloc[-1] / sales.iloc[-6]) ** (1/5) - 1) * 100, 2
                    )
                if len(sales) >= 11:
                    m['Revenue CAGR 10Y'] = round(
                        ((sales.iloc[-1] / sales.iloc[-11]) ** (1/10) - 1) * 100, 2
                    )
                if len(sales) >= 3:
                    years_available       = min(len(sales) - 1, 10)
                    base                  = sales.iloc[-years_available - 1]
                    if base > 0:
                        m['Revenue CAGR Max']   = round(
                            ((sales.iloc[-1] / base) ** (1/years_available) - 1) * 100, 2
                        )
                        m['Revenue CAGR Years'] = years_available

            # OPM / Financing Margin
            if margin_row in pl_clean.index:
                opm              = pl_clean.loc[margin_row].dropna()
                m['Avg OPM 5Y']  = round(opm.iloc[-5:].mean(),  2) if len(opm) >= 5  else round(opm.mean(), 2) if len(opm) > 0 else None
                m['Avg OPM 10Y'] = round(opm.iloc[-10:].mean(), 2) if len(opm) >= 10 else None

            # Profit CAGR
            if 'Net Profit' in pl_clean.index:
                profit = pl_clean.loc['Net Profit'].dropna()

                if len(profit) >= 6 and profit.iloc[-6] > 0:
                    m['Profit CAGR 5Y']   = round(
                        ((profit.iloc[-1] / profit.iloc[-6]) ** (1/5) - 1) * 100, 2
                    )
                if len(profit) >= 11 and profit.iloc[-11] > 0:
                    m['Profit CAGR 10Y']  = round(
                        ((profit.iloc[-1] / profit.iloc[-11]) ** (1/10) - 1) * 100, 2
                    )
                if len(profit) >= 3:
                    years_available       = min(len(profit) - 1, 10)
                    base                  = profit.iloc[-years_available - 1]
                    if base > 0:
                        m['Profit CAGR Max']   = round(
                            ((profit.iloc[-1] / base) ** (1/years_available) - 1) * 100, 2
                        )
                        m['Profit CAGR Years'] = years_available

        # ════════════════════════════════════════
        # QUARTERLY DATA → Recent momentum
        # ════════════════════════════════════════
        qr = data.get('Quarterly Results')
        if qr is not None:
            qr_clean = qr.drop(columns=['TTM'], errors='ignore')

            # Revenue
            if revenue_row in qr_clean.index:
                sales_q = qr_clean.loc[revenue_row].dropna()

                if len(sales_q) >= 4:
                    m['TTM Revenue']             = round(sales_q.iloc[-4:].sum(), 2)
                if len(sales_q) >= 5:
                    latest                       = sales_q.iloc[-1]
                    year_ago                     = sales_q.iloc[-5]
                    if year_ago > 0:
                        m['Revenue YoY Q']       = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )
                yoy_improvements = 0
                for i in range(1, min(5, len(sales_q) - 4)):
                    curr = sales_q.iloc[-i]
                    prev = sales_q.iloc[-i - 4]
                    if prev > 0 and curr > prev:
                        yoy_improvements += 1
                    else:
                        break
                m['Revenue Consecutive YoY']     = yoy_improvements

                if len(sales_q) >= 9:
                    recent_yoy                   = (sales_q.iloc[-1] - sales_q.iloc[-5]) / sales_q.iloc[-5] * 100
                    previous_yoy                 = (sales_q.iloc[-5] - sales_q.iloc[-9]) / sales_q.iloc[-9] * 100
                    m['Revenue Accelerating']    = recent_yoy > previous_yoy

            # Profit
            if 'Net Profit' in qr_clean.index:
                profit_q = qr_clean.loc['Net Profit'].dropna()

                if len(profit_q) >= 4:
                    m['TTM Profit']              = round(profit_q.iloc[-4:].sum(), 2)
                if len(profit_q) >= 5:
                    latest                       = profit_q.iloc[-1]
                    year_ago                     = profit_q.iloc[-5]
                    if year_ago > 0:
                        m['Profit YoY Q']        = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )
                m['Profit Positive Q']           = int((profit_q > 0).sum())
                m['Total Quarters']              = len(profit_q)

            # Margin
            if margin_row in qr_clean.index:
                opm_q = qr_clean.loc[margin_row].dropna()

                m['Latest OPM Q']                = opm_q.iloc[-1]  if len(opm_q) > 0 else None
                m['Avg OPM 4Q']                  = round(opm_q.iloc[-4:].mean(), 2) if len(opm_q) >= 4 else None
                if len(opm_q) >= 5:
                    m['Margin Improving']        = bool(opm_q.iloc[-1] >= opm_q.iloc[-5])

            # EPS
            if 'EPS in Rs' in qr_clean.index:
                eps_q = qr_clean.loc['EPS in Rs'].dropna()
                m['Latest EPS Q']                = eps_q.iloc[-1] if len(eps_q) > 0 else None
                if len(eps_q) >= 5:
                    latest                       = eps_q.iloc[-1]
                    year_ago                     = eps_q.iloc[-5]
                    if year_ago > 0:
                        m['EPS YoY Growth']      = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )

        # ════════════════════════════════════════
        # BALANCE SHEET → Financial structure
        # ════════════════════════════════════════
        bs = data.get('Balance Sheet')
        if bs is not None:
            if debt_row in bs.index:
                debt                             = bs.loc[debt_row].dropna()
                m['Latest Debt']                 = debt.iloc[-1]  if len(debt) > 0 else None
                if len(debt) >= 4:
                    m['Debt Reducing']           = bool(debt.iloc[-1] < debt.iloc[-4])

            if 'Reserves' in bs.index and 'Equity Capital' in bs.index:
                reserves                         = bs.loc['Reserves'].dropna()
                eq_cap                           = bs.loc['Equity Capital'].dropna()
                if len(reserves) > 0 and len(eq_cap) > 0:
                    equity                       = reserves.iloc[-1] + eq_cap.iloc[-1]
                    m['Latest Equity']           = round(equity, 2)
                    if m.get('TTM Profit') and equity > 0:
                        m['ROE']                 = round(m['TTM Profit'] / equity * 100, 2)
                    if m.get('Latest Debt') is not None and equity > 0:
                        m['Debt to Equity']      = round(m['Latest Debt'] / equity, 2)

        # ════════════════════════════════════════
        # CASH FLOWS → Cash quality
        # ════════════════════════════════════════
        cf = data.get('Cash Flows')
        if cf is not None:
            if 'Cash from Operating Activity' in cf.index:
                op_cf                            = cf.loc['Cash from Operating Activity'].dropna()
                m['Latest Operating CF']         = op_cf.iloc[-1] if len(op_cf) > 0 else None
                m['CF Positive Years']           = int((op_cf > 0).sum())
                m['CF Total Years']              = len(op_cf)
                if len(op_cf) >= 4:
                    m['CF Growing']              = bool(op_cf.iloc[-1] > op_cf.iloc[-4])

        # ════════════════════════════════════════
        # RATIOS → Efficiency
        # ════════════════════════════════════════
        ratios = data.get('Ratios')
        if ratios is not None:
            if 'ROCE %' in ratios.index:
                roce                             = ratios.loc['ROCE %'].dropna()
                m['Latest ROCE']                 = roce.iloc[-1] if len(roce) > 0 else None
                if len(roce) >= 4:
                    m['ROCE Improving']          = bool(roce.iloc[-1] > roce.iloc[-4])
                if len(roce) >= 5:
                    m['Avg ROCE 5Y']             = round(roce.iloc[-5:].mean(), 2)

            if 'ROE %' in ratios.index:
                roe_series                       = ratios.loc['ROE %'].dropna()
                m['ROE from Ratios']             = roe_series.iloc[-1] if len(roe_series) > 0 else None
                m['ROE Avg 5Y']                  = round(roe_series.iloc[-5:].mean(), 2) if len(roe_series) >= 5 else None
                m['ROE Improving']               = bool(roe_series.iloc[-1] > roe_series.iloc[-4]) if len(roe_series) >= 4 else None

        # ════════════════════════════════════════
        # SHAREHOLDING → Ownership signals
        # ════════════════════════════════════════
        sh = data.get('Shareholding Pattern')
        if sh is not None:
            if 'Promoters' in sh.index:
                promoter                         = sh.loc['Promoters'].dropna()
                m['Promoter Holding']            = promoter.iloc[-1] if len(promoter) > 0 else None
                if len(promoter) >= 5:
                    m['Promoter Change 4Q']      = round(promoter.iloc[-1] - promoter.iloc[-5], 2)
                if len(promoter) >= 9:
                    m['Promoter Change 8Q']      = round(promoter.iloc[-1] - promoter.iloc[-9], 2)

            if 'FIIs' in sh.index:
                fii                              = sh.loc['FIIs'].dropna()
                m['FII Holding']                 = fii.iloc[-1] if len(fii) > 0 else None
                if len(fii) >= 5:
                    m['FII Change 4Q']           = round(fii.iloc[-1] - fii.iloc[-5], 2)

            if 'DIIs' in sh.index:
                dii                              = sh.loc['DIIs'].dropna()
                m['DII Holding']                 = dii.iloc[-1] if len(dii) > 0 else None
                if len(dii) >= 5:
                    m['DII Change 4Q']           = round(dii.iloc[-1] - dii.iloc[-5], 2)

    except Exception as e:
        print(f"Error extracting {symbol}: {e}")

    return m


# ════════════════════════════════════════════
# RE-RUN FOR ALL 46 STOCKS
# ════════════════════════════════════════════
all_metrics = []

for symbol, data in all_stock_data.items():
    metrics = extract_all_metrics(symbol, data)
    all_metrics.append(metrics)
    print(f"✓ {symbol}")

metrics_df = pd.DataFrame(all_metrics)

# Add unified Final ROE
def get_final_roe(row):
    if pd.notna(row.get('ROE from Ratios')):
        return row['ROE from Ratios']
    elif pd.notna(row.get('ROE')):
        return row['ROE']
    return None

metrics_df['Final ROE'] = metrics_df.apply(get_final_roe, axis=1)

# Save
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

print(f"\nTotal stocks     : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")

# Check missing values
print(f"\n=== MISSING VALUES AFTER FIX ===")
missing     = metrics_df.isnull().sum()
missing_pct = (missing / len(metrics_df) * 100).round(1)

missing_report = pd.DataFrame({
    'Missing Count' : missing,
    'Missing %'     : missing_pct
}).sort_values('Missing %', ascending=False)

missing_report = missing_report[missing_report['Missing Count'] > 0]
print(missing_report.to_string())

✓ TCS
✓ INFY
✓ HDFCBANK
✓ RELIANCE
✓ WIPRO
✓ PERSISTENT
✓ MPHASIS
✓ LTTS
✓ COFORGE
✓ HAPPSTMNDS
✓ CHOLAFIN
✓ MUTHOOTFIN
✓ MANAPPURAM
✓ IIFL
✓ CREDITACC
✓ GARFIBRES
✓ SUPRAJIT
✓ HIMATSEIDE
✓ IGPL
✓ HAL
✓ BEL
✓ BHEL
✓ MIDHANI
✓ PARAS
✓ VINATIORGA
✓ CLEAN
✓ FINEORG
✓ GALAXYSURF
✓ AAVAS
✓ SUNPHARMA
✓ DIVISLAB
✓ APLLTD
✓ GRANULES
✓ IPCALAB
✓ TITAN
✓ HAVELLS
✓ VGUARD
✓ SYMPHONY
✓ WONDERLA
✓ PIIND
✓ RALLIS
✓ DHANUKA
✓ DYNPRO
✓ GULFPETRO
✓ SUVIDHAA
✓ ABSLAMC

Total stocks     : 46
Metrics per stock: 50

=== MISSING VALUES AFTER FIX ===
                         Missing Count  Missing %
ROE Avg 5Y                          39       84.8
ROE from Ratios                     39       84.8
ROE Improving                       39       84.8
Profit CAGR 10Y                     12       26.1
Revenue CAGR 10Y                    12       26.1
Avg OPM 10Y                         10       21.7
ROCE Improving                       7       15.2
Latest ROCE                          7       15.2
Avg ROCE 5Y     

In [32]:
# Save all files
import pickle

metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

survival_results = apply_survival_filter(metrics_df)
survival_results.to_csv('../data/survival_filter_results.csv', index=False)

with open('../data/raw_stock_data.pkl', 'wb') as f:
    pickle.dump(all_stock_data, f)

passed = survival_results[survival_results['Passed Survival'] == True]
failed = survival_results[survival_results['Passed Survival'] == False]

print(f"Total stocks     : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")
print(f"Passed survival  : {len(passed)}")
print(f"Failed survival  : {len(failed)}")
print(f"All files saved  ✅")

Total stocks     : 46
Metrics per stock: 50
Passed survival  : 35
Failed survival  : 11
All files saved  ✅


In [33]:
# Check ROE vs ROE from Ratios vs Final ROE
print(metrics_df[['Symbol', 'Sector', 'ROE', 
                   'ROE from Ratios', 'Final ROE']].to_string())

        Symbol                          Sector    ROE  ROE from Ratios  Final ROE
0          TCS          Information Technology  45.07              NaN      45.07
1         INFY          Information Technology  27.10              NaN      27.10
2     HDFCBANK              Financial Services  14.36             14.0      14.00
3     RELIANCE     Oil, Gas & Consumable Fuels  11.15              NaN      11.15
4        WIPRO          Information Technology  15.58              NaN      15.58
5   PERSISTENT          Information Technology  24.22              NaN      24.22
6      MPHASIS          Information Technology  18.70              NaN      18.70
7         LTTS          Information Technology  20.33              NaN      20.33
8      COFORGE          Information Technology  20.09              NaN      20.09
9   HAPPSTMNDS          Information Technology  11.36              NaN      11.36
10    CHOLAFIN              Financial Services  18.69             20.0      20.00
11  MUTHOOTFIN  